In [1]:
#Dependencias para usar 
from selenium import webdriver
import time
import os
import time
import requests
from bs4 import BeautifulSoup
from pprint import pprint
import pandas as pd
import json
from lxml import html
import re
import csv
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service #para genrar un objeto de tipo Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    WebDriverException,
    )
from webdriver_manager.chrome import ChromeDriverManager #descargar lo
import pandas as pd

In [2]:
options = Options()
options.add_argument('--ignore-certificate-errors')
options.add_argument('--disable-notifications')


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=options)
driver.get("https://www.google.com/")
#primera busqueda
search = driver.find_element(By.NAME, value="q")
search.send_keys("Restaurantes CDMX")
search.send_keys(Keys.ENTER)

time.sleep(5)
avanzar = driver.find_elements(By.LINK_TEXT,'Más lugares')
for item in avanzar:
  item.click()

[WDM] - Downloading: 100%|██████████| 8.08M/8.08M [00:00<00:00, 10.3MB/s]


In [3]:
def res_hasta_pag_n (n,text):
    resultados = []
    for i in range(1,n+1):
      otr = driver.find_elements(By.LINK_TEXT,'{}'.format(str(i)))
      for item in otr:
        item.click()
        time.sleep(5)
      nueva_pagina = [contenido.text for contenido in driver.find_elements(By.CLASS_NAME,text)]
      resultados.append(nueva_pagina)
    print("se terminó la busqueda")
    return resultados

In [4]:
resultados_busqueda = res_hasta_pag_n(20,"rllt__details")

se terminó la busqueda


In [5]:
import itertools
import copy #librería par ahacer copias 

RB = copy.copy(resultados_busqueda) #guardamos una lista del objeto 

In [6]:
print(len(RB))

20


In [7]:
print(RB)

[['Taboo | Restaurante Mediterráneo en Polanco\nAnuncio·4.2\n(575) · Restaurante de cocina mediterránea\nCDMX\nAbierto ⋅ Cierra a las 02:00\nConsumo en el lugar·\nPara llevar', 'Cambalache Oasis Coyoacán\n4.6\n(2.6 K) · $$$ · Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Los Danzantes Coyoacán\n4.4\n(3.6 K) · $$$ · Mexicana\nParque Centenario 12\nConsumo en el lugar·\nRetiros en la puerta·\nNo ofrece servicio de entrega a domicilio', 'BISTRO 83\n4.5\n(1.3 K) · $$$ · Restaurante\nC. de la Amargura 17\nConsumo en el lugar·\nPara llevar·\nEntrega a domicilio', 'San Ángel Inn\n4.6\n(8.1 K) · $$$ · Mexicana\nDiego Rivera 50\nConsumo en el lugar·\nPara llevar·\nNo ofrece servicio de entrega a domicilio', 'Sud 777\n4.4\n(2.7 K) · $$$ · Mexicana\nBlvrd de la Luz 777\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Carlota\n4.4\n(791) · $$$ · Mexicana\nDel Carmen 4\nConsumo en el lugar·\nNo ofrece servicio d

In [8]:
#aplanamos la lista 
RB = list(itertools.chain.from_iterable(resultados_busqueda)) 

In [9]:
print(RB)

['Taboo | Restaurante Mediterráneo en Polanco\nAnuncio·4.2\n(575) · Restaurante de cocina mediterránea\nCDMX\nAbierto ⋅ Cierra a las 02:00\nConsumo en el lugar·\nPara llevar', 'Cambalache Oasis Coyoacán\n4.6\n(2.6 K) · $$$ · Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Los Danzantes Coyoacán\n4.4\n(3.6 K) · $$$ · Mexicana\nParque Centenario 12\nConsumo en el lugar·\nRetiros en la puerta·\nNo ofrece servicio de entrega a domicilio', 'BISTRO 83\n4.5\n(1.3 K) · $$$ · Restaurante\nC. de la Amargura 17\nConsumo en el lugar·\nPara llevar·\nEntrega a domicilio', 'San Ángel Inn\n4.6\n(8.1 K) · $$$ · Mexicana\nDiego Rivera 50\nConsumo en el lugar·\nPara llevar·\nNo ofrece servicio de entrega a domicilio', 'Sud 777\n4.4\n(2.7 K) · $$$ · Mexicana\nBlvrd de la Luz 777\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Carlota\n4.4\n(791) · $$$ · Mexicana\nDel Carmen 4\nConsumo en el lugar·\nNo ofrece servicio de

In [10]:
print(len(RB))

397


In [11]:
RB = [re.split(r'·\n| · ',item) for item in RB] 

In [12]:
print(RB)

[['Taboo | Restaurante Mediterráneo en Polanco\nAnuncio·4.2\n(575)', 'Restaurante de cocina mediterránea\nCDMX\nAbierto ⋅ Cierra a las 02:00\nConsumo en el lugar', 'Para llevar'], ['Cambalache Oasis Coyoacán\n4.6\n(2.6 K)', '$$$', 'Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\nConsumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Los Danzantes Coyoacán\n4.4\n(3.6 K)', '$$$', 'Mexicana\nParque Centenario 12\nConsumo en el lugar', 'Retiros en la puerta', 'No ofrece servicio de entrega a domicilio'], ['BISTRO 83\n4.5\n(1.3 K)', '$$$', 'Restaurante\nC. de la Amargura 17\nConsumo en el lugar', 'Para llevar', 'Entrega a domicilio'], ['San Ángel Inn\n4.6\n(8.1 K)', '$$$', 'Mexicana\nDiego Rivera 50\nConsumo en el lugar', 'Para llevar', 'No ofrece servicio de entrega a domicilio'], ['Sud 777\n4.4\n(2.7 K)', '$$$', 'Mexicana\nBlvrd de la Luz 777\nConsumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Carlota\n4.4\n(791)', '$$$', 'Mexicana\nDel Carmen 4\nConsum

In [13]:
#para cada lista de cadenas de caracrteres sustituimos los patrones ' ⋅ ',': ⋅ ' por '\n'
RB = [[re.sub(r' ⋅ |: ⋅ ','\n',cadena) for cadena in lista_cadena] for lista_cadena in RB ]

In [14]:
#dentro de cada lista la cadena de caracteres se separa por 
RB = [[re.split(r'\n',cadena) for cadena in lista_cadena] for lista_cadena in RB]

In [15]:
print(RB)

[[['Taboo | Restaurante Mediterráneo en Polanco', 'Anuncio·4.2', '(575)'], ['Restaurante de cocina mediterránea', 'CDMX', 'Abierto', 'Cierra a las 02:00', 'Consumo en el lugar'], ['Para llevar']], [['Cambalache Oasis Coyoacán', '4.6', '(2.6 K)'], ['$$$'], ['Argentina', 'Avenida Miguel Ángel de Quevedo 227 Local R-03-04', 'Consumo en el lugar'], ['Para llevar'], ['Entrega sin contacto']], [['Los Danzantes Coyoacán', '4.4', '(3.6 K)'], ['$$$'], ['Mexicana', 'Parque Centenario 12', 'Consumo en el lugar'], ['Retiros en la puerta'], ['No ofrece servicio de entrega a domicilio']], [['BISTRO 83', '4.5', '(1.3 K)'], ['$$$'], ['Restaurante', 'C. de la Amargura 17', 'Consumo en el lugar'], ['Para llevar'], ['Entrega a domicilio']], [['San Ángel Inn', '4.6', '(8.1 K)'], ['$$$'], ['Mexicana', 'Diego Rivera 50', 'Consumo en el lugar'], ['Para llevar'], ['No ofrece servicio de entrega a domicilio']], [['Sud 777', '4.4', '(2.7 K)'], ['$$$'], ['Mexicana', 'Blvrd de la Luz 777', 'Consumo en el lugar'],

In [16]:
#se aplana cada lista que cadenas (que fue separada) 
RB = [list(itertools.chain.from_iterable(
  [[cadena for cadena in lista_cadena_separada]for lista_cadena_separada 
  in lista_cadena_RB])) for lista_cadena_RB in RB] 


In [17]:
print(RB) 
#los datos aparecen de la forma 

[['Taboo | Restaurante Mediterráneo en Polanco', 'Anuncio·4.2', '(575)', 'Restaurante de cocina mediterránea', 'CDMX', 'Abierto', 'Cierra a las 02:00', 'Consumo en el lugar', 'Para llevar'], ['Cambalache Oasis Coyoacán', '4.6', '(2.6 K)', '$$$', 'Argentina', 'Avenida Miguel Ángel de Quevedo 227 Local R-03-04', 'Consumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Los Danzantes Coyoacán', '4.4', '(3.6 K)', '$$$', 'Mexicana', 'Parque Centenario 12', 'Consumo en el lugar', 'Retiros en la puerta', 'No ofrece servicio de entrega a domicilio'], ['BISTRO 83', '4.5', '(1.3 K)', '$$$', 'Restaurante', 'C. de la Amargura 17', 'Consumo en el lugar', 'Para llevar', 'Entrega a domicilio'], ['San Ángel Inn', '4.6', '(8.1 K)', '$$$', 'Mexicana', 'Diego Rivera 50', 'Consumo en el lugar', 'Para llevar', 'No ofrece servicio de entrega a domicilio'], ['Sud 777', '4.4', '(2.7 K)', '$$$', 'Mexicana', 'Blvrd de la Luz 777', 'Consumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Carlota',

In [18]:
#Función para convertir a tipo float
def convierte_float(item): 
    try: 
      fitem = float(item)
      return fitem 
    except ValueError:
      return item  


In [19]:
RB = [list(map(convierte_float, cadena)) for cadena in RB ]

In [20]:
print(RB)

[['Taboo | Restaurante Mediterráneo en Polanco', 'Anuncio·4.2', '(575)', 'Restaurante de cocina mediterránea', 'CDMX', 'Abierto', 'Cierra a las 02:00', 'Consumo en el lugar', 'Para llevar'], ['Cambalache Oasis Coyoacán', 4.6, '(2.6 K)', '$$$', 'Argentina', 'Avenida Miguel Ángel de Quevedo 227 Local R-03-04', 'Consumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Los Danzantes Coyoacán', 4.4, '(3.6 K)', '$$$', 'Mexicana', 'Parque Centenario 12', 'Consumo en el lugar', 'Retiros en la puerta', 'No ofrece servicio de entrega a domicilio'], ['BISTRO 83', 4.5, '(1.3 K)', '$$$', 'Restaurante', 'C. de la Amargura 17', 'Consumo en el lugar', 'Para llevar', 'Entrega a domicilio'], ['San Ángel Inn', 4.6, '(8.1 K)', '$$$', 'Mexicana', 'Diego Rivera 50', 'Consumo en el lugar', 'Para llevar', 'No ofrece servicio de entrega a domicilio'], ['Sud 777', 4.4, '(2.7 K)', '$$$', 'Mexicana', 'Blvrd de la Luz 777', 'Consumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Carlota', 4.4, '(79

In [23]:
def quita_parentesis(item):
  try: 
     sin_par = re.sub(r'\(|\)','',item)
     return sin_par
  except TypeError:
    return item   

In [24]:
RB = [list(map(quita_parentesis,cadena)) for cadena in RB]

In [25]:
print(RB)

[['Balcón del Zócalo', 4.5, '5.8 K', 'Restaurante mexicano', 'Cuauhtémoc, CDMX', 'Entrega a domicilio'], ['Entre Gauchos', 4.3, '910', 'Restaurante argentino', 'Tlalnepantla, Estado de México', 'Retiros en la puerta', 'Entrega sin contacto'], ['Cambalache Oasis Coyoacán', 4.6, '2.6 K', 'Argentina', 'Avenida Miguel Ángel de Quevedo 227 Local R-03-04', 'Consumo en el lugar', 'Para llevar', 'Entrega sin contacto'], ['Los Danzantes Coyoacán', 4.4, '3.6 K', 'Mexicana', 'Parque Centenario 12', 'Cierra pronto', '23:00', 'Consumo en el lugar', 'Retiros en la puerta', 'No ofrece servicio de entrega a domicilio'], ['BISTRO 83', 4.5, '1.3 K', 'Restaurante', 'C. de la Amargura 17', 'Cierra pronto', '23:00', 'Consumo en el lugar', 'Para llevar', 'Entrega a domicilio'], ['San Ángel Inn', 4.6, '8.1 K', 'Mexicana', 'Diego Rivera 50', 'Consumo en el lugar', 'Para llevar', 'No ofrece servicio de entrega a domicilio'], ['Sud 777', 4.4, '2.7 K', 'Mexicana', 'Blvrd de la Luz 777', 'Consumo en el lugar', 'P

In [21]:
#calculamos la lista de cadenas que tiene mas elementos  
num_atributos = [len(lista_cadena) for lista_cadena in RB ]
print(num_atributos)

[9, 9, 9, 9, 9, 9, 8, 9, 7, 9, 9, 9, 9, 8, 9, 8, 9, 9, 9, 8, 9, 9, 9, 9, 8, 9, 9, 9, 9, 8, 8, 8, 8, 9, 9, 9, 9, 8, 7, 8, 9, 9, 9, 9, 8, 7, 9, 9, 9, 8, 9, 6, 9, 8, 8, 9, 9, 9, 7, 8, 9, 8, 8, 9, 9, 9, 9, 9, 8, 9, 9, 9, 9, 9, 8, 7, 10, 8, 9, 9, 8, 8, 8, 7, 9, 8, 8, 9, 7, 9, 9, 9, 9, 9, 10, 7, 9, 8, 7, 7, 9, 9, 8, 9, 9, 9, 9, 9, 7, 7, 7, 7, 9, 9, 9, 8, 9, 8, 9, 9, 9, 9, 7, 8, 11, 7, 7, 8, 9, 9, 8, 9, 9, 8, 8, 7, 7, 8, 9, 9, 9, 9, 7, 9, 7, 7, 8, 9, 8, 7, 9, 9, 8, 9, 9, 8, 9, 8, 8, 8, 7, 8, 9, 9, 8, 8, 9, 9, 6, 9, 8, 6, 8, 9, 8, 8, 8, 9, 9, 8, 8, 9, 9, 9, 8, 9, 8, 9, 7, 7, 8, 9, 8, 9, 9, 8, 8, 9, 6, 9, 9, 9, 9, 7, 9, 8, 9, 9, 9, 9, 7, 8, 7, 7, 6, 5, 9, 7, 6, 9, 8, 7, 8, 7, 8, 7, 7, 9, 8, 9, 9, 7, 9, 10, 7, 9, 8, 8, 8, 8, 9, 5, 5, 5, 6, 6, 5, 6, 4, 5, 6, 6, 5, 4, 6, 6, 6, 6, 6, 6, 6, 7, 9, 9, 9, 8, 9, 8, 7, 8, 8, 7, 7, 9, 7, 9, 8, 7, 9, 8, 8, 6, 6, 6, 6, 6, 5, 5, 6, 5, 6, 5, 6, 7, 5, 6, 5, 5, 6, 6, 6, 7, 6, 5, 6, 6, 6, 5, 6, 5, 5, 6, 9, 6, 5, 6, 5, 6, 5, 6, 5, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6, 5,

In [ ]:
def fila_aceptable(lst):
  for i in len(lst):
   pass